# Una rete profonda con NumPy

Materiale originale **AI Lab**, non una lezione Kaggle. Codice in inglese; attività e spiegazioni in italiano.

Esegui le celle in ordine con **Maiusc + Invio**. Salva una copia prima di modificare il codice.


## Obiettivi e requisiti
NumPy installato. Riconoscere neuroni, due strati nascosti, sigmoid, cross-entropy e gradiente. Il problema sintetico assegna classe 1 ai punti nei quadranti con coordinate dello stesso segno: una sola retta non separa le classi. CPU sufficiente.


In [ ]:
import numpy as np

rng = np.random.default_rng(42)
features = rng.uniform(-1, 1, (800, 2)).astype('float32')
targets = ((features[:, 0] * features[:, 1]) > 0).astype('float32').reshape(-1, 1)
order = rng.permutation(len(features))
features, targets = features[order], targets[order]
X_train, X_valid, X_test = features[:480], features[480:640], features[640:]
y_train, y_valid, y_test = targets[:480], targets[480:640], targets[640:]
assert not np.isnan(features).any()
print('Training shape:', X_train.shape, 'Target shape:', y_train.shape)


## Addestramento reale
La rete ha due strati nascosti (16 e 8 unità tanh) e un’uscita sigmoid. Il codice applica la backpropagation e minibatch SGD. È materiale originale AI Lab, pensato per vedere i calcoli che Keras automatizza.


In [ ]:
# Two hidden layers learn a nonlinear XOR-like rule.
W1 = rng.normal(0, 0.5, (2, 16))
b1 = np.zeros((1, 16))
W2 = rng.normal(0, 0.3, (16, 8))
b2 = np.zeros((1, 8))
W3 = rng.normal(0, 0.3, (8, 1))
b3 = np.zeros((1, 1))
parameters = [W1, b1, W2, b2, W3, b3]
learning_rate = 0.15
batch_size = 32
history = []
best_loss = float('inf')
best_parameters = [value.copy() for value in parameters]
wait = 0


def forward(X):
    hidden1 = np.tanh(X @ W1 + b1)
    hidden2 = np.tanh(hidden1 @ W2 + b2)
    probabilities = 1 / (1 + np.exp(-np.clip(hidden2 @ W3 + b3, -40, 40)))
    return hidden1, hidden2, probabilities


def cross_entropy(actual, predicted):
    p = np.clip(predicted, 1e-8, 1-1e-8)
    return float(-np.mean(actual*np.log(p) + (1-actual)*np.log(1-p)))


for epoch in range(400):
    permutation = rng.permutation(len(X_train))
    for start in range(0, len(X_train), batch_size):
        selection = permutation[start:start+batch_size]
        X, y = X_train[selection], y_train[selection]
        h1, h2, probability = forward(X)
        d3 = (probability-y) / len(X)
        d2 = (d3 @ W3.T) * (1-h2*h2)
        d1 = (d2 @ W2.T) * (1-h1*h1)
        gradients = [X.T @ d1, d1.sum(axis=0, keepdims=True),
                     h1.T @ d2, d2.sum(axis=0, keepdims=True),
                     h2.T @ d3, d3.sum(axis=0, keepdims=True)]
        for parameter, gradient in zip(parameters, gradients):
            parameter -= learning_rate * gradient
    train_loss = cross_entropy(y_train, forward(X_train)[2])
    valid_loss = cross_entropy(y_valid, forward(X_valid)[2])
    history.append([train_loss, valid_loss])
    if valid_loss < best_loss - 0.0001:
        best_loss = valid_loss
        best_parameters = [value.copy() for value in parameters]
        wait = 0
    else:
        wait += 1
    if wait >= 25:
        break
for parameter, best in zip(parameters, best_parameters):
    parameter[:] = best
accuracy = float(np.mean((forward(X_test)[2] >= 0.5) == y_test))
assert np.isfinite(history).all()
assert history[-1][0] < history[0][0]
assert accuracy >= 0.85
print('Epochs:', len(history), 'Test accuracy:', accuracy)
print('Initial / final training loss:', history[0][0], history[-1][0])


## Leggere le curve
Ogni riga di `history` contiene loss di training e validazione. Un grafico SVG senza ulteriori dipendenze viene salvato qui sotto.


In [ ]:
from pathlib import Path

points = " ".join(f"{20+i*560/max(1,len(history)-1):.2f},{180-150*min(1,row[1]):.2f}" for i,row in enumerate(history))
svg = f'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 600 210"><rect width="600" height="210" fill="white"/><polyline fill="none" stroke="teal" stroke-width="2" points="{points}"/><text x="20" y="205">Validation cross-entropy by epoch</text></svg>'
Path("learning-curve.svg").write_text(svg)
print("Saved learning-curve.svg")


## Esercizi
1. Cambia il learning rate a 0.03 e confronta.
2. Riduci la prima dimensione nascosta e adegua W1, b1 e W2.
3. Spiega perché separiamo y_valid da y_test.
4. Individua nel codice la derivata di tanh e l’aggiornamento dei pesi.

## Riepilogo
Le non linearità permettono una frontiera complessa. Il gradiente aggiorna i pesi e la validazione guida l’arresto.
